# tuples
Tuples are fixed-size ordered containers. In real systems they are useful when the shape is stable and immutability communicates intent: coordinates, keys, pairs, records, and return bundles. Their value is less about syntax and more about semantic stability.

## 1. Problem First
Why does this matter in real systems?
- Function returns often bundle multiple values in tuples.
- Dictionary keys frequently use tuples for compound identity.
- Immutability reduces accidental structural mutation.

In [ ]:
location = (19.07, 72.87)
print(location[0])
print(len(location))

## 2. Minimal Concept
Core syntax:
- Create with `(a, b)`
- Single-item tuple needs a trailing comma: `(a,)`
- Access by index and unpack by position
- Tuples do not support item reassignment

## 3. Mental Model
How Python actually behaves
A tuple is immutable in structure, which means its length and element bindings cannot change. But immutability is shallow: a tuple can still contain mutable objects whose contents can change. That distinction matters when people assume tuples are always “fully safe.”

In [ ]:
record = ("api", [200, 201])
record[1].append(500)
print(record)

## 4. Internal Mechanics
Tuple immutability means there are no mutating methods like `append`. Python can use tuples as dictionary keys if all contained values are hashable. That is a major reason tuples matter in lookup-heavy systems.

In [ ]:
import dis

def pair(service, region):
    return (service, region)

dis.dis(pair)
cache = {("api", "us-east-1"): "healthy"}
print(cache[("api", "us-east-1")])

## 5. Edge Cases
Where it breaks:
- `(1)` is just `1`, not a tuple.
- Tuples containing lists are not hashable.
- Structural immutability does not protect nested mutable objects.

In [ ]:
single = (1)
real_single = (1,)
print(type(single))
print(type(real_single))

try:
    lookup = {([1, 2],): "bad"}
except TypeError as exc:
    print(type(exc).__name__)

## 6. Performance Thinking
- Tuple indexing is O(1).
- Tuples are typically a bit lighter than lists because they do not support resizing.
- Their main performance advantage is safe hashability when used as keys, not magical speed in all cases.

## 7. Real Use Case
A cache for service health might key entries by `(service, region)` because the pair together forms the stable identity.

In [ ]:
health = {
    ("api", "us"): "ok",
    ("worker", "eu"): "degraded"
}
print(health[("api", "us")])

## 8. Anti-Pattern
What beginners do wrong:
- Use tuples when named structure would be clearer.
- Assume nested mutable contents are protected.
- Forget the trailing comma for single-item tuples.

In [ ]:
response = (200, "ok", {"latency": 120})
response[2]["latency"] = 999
print(response)

## 9. Interview Signals
Questions FAANG asks:
- Why can tuples be dict keys while lists cannot?
- Is a tuple always fully immutable?
- When would you choose a tuple over a list or dataclass?
- What is the single-element tuple syntax trap?

## 10. Exercise (Non-trivial)
Design a cache key strategy for a service using multiple dimensions like service name, region, and API version. Decide whether a tuple is the right representation, how you would avoid collisions, and what happens if one dimension is itself mutable.

In [ ]:
def make_cache_key(service, region, version):
    return (service, region, version)

# Task:
# 1. Explain why tuple works well here.
# 2. Identify what would break hashability.
# 3. Compare this to using a nested dictionary.